# Project Forge — Gerador de Sprites

**Instruções:**
1. Clique em **Runtime → Change runtime type → T4 GPU**
2. Clique em **Runtime → Run all** (ou **Executar tudo**)
3. Role até o final e digite o que deseja criar
4. A sprite vai aparecer na tela!
---

In [ ]:
# @title 1. Instalar ComfyUI (só esperar)
import os, sys, subprocess, time, json, urllib.request, threading, base64, IPython.display
from google.colab import output

COMFY_DIR = "/content/ComfyUI"

if not os.path.exists(COMFY_DIR):
    print("Instalando ComfyUI...")
    !apt-get update -qq -y && apt-get install -qq -y libgl1-mesa-glx libglib2.0-0 > /dev/null 2>&1
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git "{COMFY_DIR}" 2>&1 | tail -1
    !pip install -q -r "{COMFY_DIR}/requirements.txt" 2>&1 | tail -1
    print("ComfyUI instalado!")
else:
    print("ComfyUI ja instalado")

In [ ]:
# @title 2. Baixar modelo (aguarde o download)
MODEL_NAME = "dreamshaper_8.safetensors"
MODEL_PATH = f"{COMFY_DIR}/models/checkpoints/{MODEL_NAME}"

if not os.path.exists(MODEL_PATH):
    print("Baixando modelo Dreamshaper (~2GB)...")
    !wget -q --show-progress -O "{MODEL_PATH}" \
        "https://civitai.com/api/download/models/128713?type=Model&format=SafeTensor&size=pruned&fp=fp16"
    print("Modelo baixado!")
else:
    print("Modelo ja existe")

In [ ]:
# @title 3. Iniciar servidor ComfyUI
log_file = open('/content/comfyui.log', 'w')
server = subprocess.Popen(
    [sys.executable, "main.py", "--port=8188", "--listen=0.0.0.0"],
    cwd=COMFY_DIR, stdout=log_file, stderr=subprocess.STDOUT, text=True
)

print("Aguardando servidor...")
for i in range(120):
    try:
        urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/system_stats"), timeout=2)
        print(f"Servidor pronto! ({i+1}s)")
        break
    except:
        time.sleep(1)
else:
    print("ERRO: servidor nao iniciou")
    raise SystemExit()

In [ ]:
# @title 4. Abrir túnel público
# Inicia Cloudflare Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

tunnel_log = open('/content/tunnel.log', 'w')
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

tunnel_url = None
def read_output():
    global tunnel_url
    for line in tunnel.stdout:
        print(f"[tunnel] {line.strip()}")
        if "https://" in line and ".trycloudflare.com" in line:
            for word in line.split():
                if "https://" in word and ".trycloudflare.com" in word:
                    tunnel_url = word.rstrip(".")

threading.Thread(target=read_output, daemon=True).start()

for _ in range(30):
    if tunnel_url:
        break
    time.sleep(1)

if tunnel_url:
    print(f"\nTUNEL ABERTO: {tunnel_url}")
else:
    print("Aviso: tunel pode estar lento para conectar...")
    tunnel_url = "http://127.0.0.1:8188"

---
## 🎮 GERAR SPRITE
---

In [ ]:
# @title 5. GERAR SPRITE 🎮
# -----------------------------------------------------------
# DIGITE O QUE DESEJA CRIAR NA LINHA ABAIXO:
# -----------------------------------------------------------
prompt_usuario = "fire dragon, pixel art, rpg game character, 16bit"  # <-- MUDE AQUI
# -----------------------------------------------------------

print(f"Gerando: {prompt_usuario}")
print()

# Workflow igual ao do Project Forge
workflow = {
    "3": {
        "class_type": "KSampler",
        "inputs": {
            "seed": 42,
            "steps": 25,
            "cfg": 7,
            "sampler_name": "euler",
            "scheduler": "normal",
            "denoise": 1,
            "model": ["4", 0],
            "positive": ["6", 0],
            "negative": ["7", 0],
            "latent_image": ["5", 0]
        }
    },
    "4": {
        "class_type": "CheckpointLoaderSimple",
        "inputs": {"ckpt_name": "dreamshaper_8.safetensors"}
    },
    "5": {
        "class_type": "EmptyLatentImage",
        "inputs": {"width": 512, "height": 512, "batch_size": 1}
    },
    "6": {
        "class_type": "CLIPTextEncode",
        "inputs": {
            "text": prompt_usuario + ", pixel art, game sprite, 16bit, rpg character, high quality, detailed, sharp pixels",
            "clip": ["4", 1]
        }
    },
    "7": {
        "class_type": "CLIPTextEncode",
        "inputs": {
            "text": "blurry, low quality, deformed, ugly, bad anatomy, watermark, text, realistic, photo",
            "clip": ["4", 1]
        }
    },
    "8": {
        "class_type": "VAEDecode",
        "inputs": {"samples": ["3", 0], "vae": ["4", 2]}
    },
    "9": {
        "class_type": "SaveImage",
        "inputs": {"filename_prefix": "forge_sprite", "images": ["8", 0]}
    }
}

# Enviar para o ComfyUI
data = json.dumps({"prompt": workflow}).encode("utf-8")
req = urllib.request.Request(
    f"{tunnel_url}/prompt", data=data,
    headers={"Content-Type": "application/json"}
)
resp = urllib.request.urlopen(req)
result = json.loads(resp.read().decode())
prompt_id = result["prompt_id"]
print(f"Job enviado! ID: {prompt_id}")

# Aguardar resultado
print("Aguardando geracao...")
for _ in range(300):
    time.sleep(1)
    req = urllib.request.Request(f"{tunnel_url}/history/{prompt_id}")
    resp = urllib.request.urlopen(req)
    history = json.loads(resp.read().decode())
    if prompt_id in history:
        outputs = history[prompt_id].get("outputs", {})
        images = []
        for node_id in outputs:
            images.extend(outputs[node_id].get("images", []))
        if images:
            print("Sprite gerada!")
            # Baixar e mostrar
            img_info = images[0]
            params = urllib.parse.urlencode({
                "filename": img_info["filename"],
                "subfolder": img_info.get("subfolder", ""),
                "type": img_info.get("type", "output")
            })
            req = urllib.request.Request(f"{tunnel_url}/view?{params}")
            img_data = urllib.request.urlopen(req).read()

            # Salvar
            with open("/content/sprite_gerada.png", "wb") as f:
                f.write(img_data)

            # Mostrar
            from IPython.display import display, Image
            display(Image(data=img_data))
            print(f"\nSprite salva em: /content/sprite_gerada.png")
            print(f"Prompt usado: {prompt_usuario}")
            break
else:
    print("Tempo esgotado. Tente novamente.")